# H3-4 — Cryptanalyse du Hill cipher

**Objectif** : casser un chiffrement de Hill 2×2 avec deux approches CP-SAT :
1. Attaque à texte clair connu (algèbre linéaire mod 26)
2. Attaque texte chiffré seul (optimisation bigrammes)

## Plan
1. [Le chiffrement de Hill](#1)
2. [Algèbre mod 26 et invertibilité](#2)
3. [Attaque à texte clair connu — CP-SAT](#3)
4. [Attaque texte chiffré seul — CP-SAT bigrammes](#4)
5. [Benchmark et comparaison](#5)

In [ ]:
import sys, os, time, random
import string
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from core.ciphers.hill              import (generate_random_key, encrypt, decrypt,
                                             key_accuracy, known_plaintext_attack,
                                             _matrix_inv_mod26)
from core.linguistics.frequency_analysis import (
    clean_text, letter_frequencies, bigram_log_probs,
    index_of_coincidence, score_text
)
from core.solvers.cp_hill           import (solve_hill_known_plaintext,
                                             solve_hill_ciphertext_only)

ALPHABET = string.ascii_uppercase
random.seed(42)

In [ ]:
with open(os.path.join('..', 'data', 'french_reference.txt'), encoding='utf-8') as f:
    CORPUS = f.read()

BIGRAM_LP   = bigram_log_probs(CORPUS)
LETTER_FREQ = letter_frequencies(CORPUS)
print(f"Corpus : {len(clean_text(CORPUS))} lettres")

<a id='1'></a>
---
## 1. Le chiffrement de Hill

### Principe

Chiffrement **linéaire par blocs** : les lettres sont chiffrées par groupes de n selon une multiplication matricielle mod 26.

```
Bloc de clair  : [p1, p2]ᵀ
Clé            : K = [[k00, k01], [k10, k11]]  (2×2, inversible mod 26)
Chiffré        : [c1, c2]ᵀ = K × [p1, p2]ᵀ  (mod 26)

Exemple : K = [[6, 24], [1, 13]], bloc 'JU' = [9, 20]
  c1 = (6×9 + 24×20) mod 26 = (54 + 480) mod 26 = 534 mod 26 = 14 → O
  c2 = (1×9 + 13×20) mod 26 = (9 + 260)  mod 26 = 269 mod 26 = 9  → J
```

**Condition d'inversibilité** : `gcd(det(K) mod 26, 26) = 1`  
Les valeurs det valides : {1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25}

**Espace** : 26⁴ = 456 976 matrices possibles, dont environ 157 248 inversibles.

**Vulnérabilité** : la structure linéaire permet une attaque à texte clair connu avec 2 paires seulement.

In [ ]:
# ── Texte de test ─────────────────────────────────────────────────────────
TEST_PLAIN = clean_text("""
    La justice est lame de la societe comme le feu est lame des forges.
    On ne peut pas vivre sans justice comme on ne peut pas vivre sans pain.
    Jean Valjean etait un homme de taille forte et robuste dans la fleur de l age.
    Paris est une ville immense ou se melent toutes les races et toutes les fortunes.
    La France est un grand pays avec une longue histoire et une culture riche.
""")

TRUE_KEY   = generate_random_key()
CIPHERTEXT = encrypt(TEST_PLAIN, TRUE_KEY)

det = (TRUE_KEY[0][0]*TRUE_KEY[1][1] - TRUE_KEY[0][1]*TRUE_KEY[1][0]) % 26
print(f"Texte clair : {len(TEST_PLAIN)} lettres")
print(f"Clé K = {TRUE_KEY}")
print(f"det(K) mod 26 = {det}  (doit être impair et non divisible par 13)")
print(f"Chiffré (extrait): {CIPHERTEXT[:80]}")

# Vérification round-trip (decrypt retourne le texte paddé si longueur impaire)
assert decrypt(CIPHERTEXT, TRUE_KEY)[:len(TEST_PLAIN)] == TEST_PLAIN
print("\nencrypt/decrypt OK")

<a id='2'></a>
---
## 2. Algèbre mod 26 et invertibilité

### Inverse modulaire d'une matrice 2×2

```
K⁻¹ = (1/det) × [[d, -b], [-c, a]]  (mod 26)

où  det = ad - bc  (mod 26)
    1/det = inverse modulaire de det mod 26
```

### IC du Hill cipher

Hill est un chiffrement par **blocs** : l'IC des lettres individuelles est modifié  
(les lettres ne sont plus substituées indépendamment).  

→ On ne peut pas utiliser l'analyse de fréquence directement.  
→ La structure linéaire est la faille principale.

In [ ]:
# ── Propriétés statistiques ───────────────────────────────────────────────
print("=== Clé K ===")
print(f"K     = {TRUE_KEY}")
print(f"K⁻¹   = {_matrix_inv_mod26(TRUE_KEY)}")

# Vérification : K × K⁻¹ ≡ I (mod 26)
K     = TRUE_KEY
K_inv = _matrix_inv_mod26(TRUE_KEY)
I_check = [[(K[i][0]*K_inv[0][j] + K[i][1]*K_inv[1][j]) % 26 for j in range(2)] for i in range(2)]
print(f"K × K⁻¹ mod 26 = {I_check}  (doit être [[1,0],[0,1]])")

print(f"\nIC texte clair  : {index_of_coincidence(TEST_PLAIN):.4f}")
print(f"IC texte chiffré: {index_of_coincidence(CIPHERTEXT):.4f}")
print(f"Score bigrammes clair  : {score_text(TEST_PLAIN, BIGRAM_LP):.0f}")
print(f"Score bigrammes chiffré: {score_text(CIPHERTEXT, BIGRAM_LP):.0f}")

# Distribution des fréquences
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (txt, title, color) in zip(axes, [
    (TEST_PLAIN, 'Texte clair',        'seagreen'),
    (CIPHERTEXT, 'Hill chiffré (2×2)', 'steelblue'),
]):
    freqs = letter_frequencies(txt)
    ax.bar(list(ALPHABET), [freqs[l] for l in ALPHABET], color=color, alpha=0.85)
    ax.set_title(title)
    ax.tick_params(axis='x', labelsize=7)
plt.suptitle('Hill cipher — IC similaire mais distribution légèrement modifiée', y=1.02)
plt.tight_layout()
os.makedirs('../examples', exist_ok=True)
plt.savefig('../examples/hill_freq.png', dpi=120, bbox_inches='tight')
plt.show()

<a id='3'></a>
---
## 3. Attaque à texte clair connu — CP-SAT

### Principe algébrique

Avec 2 paires (clair, chiffré) de blocs de 2 lettres :

```
[C1 | C2] = K × [P1 | P2]   (mod 26)
→  K = [C1 | C2] × [P1 | P2]⁻¹  (mod 26)
```

**Condition** : `[P1 | P2]` doit être invertible mod 26.

### Modèle CP-SAT

```
Variables   : K[i][j] ∈ [0..25]
Contraintes : K[row][0]*p0 + K[row][1]*p1 ≡ c_row  (mod 26)  pour chaque paire
              gcd(det(K), 26) = 1  (inversibilité)
```

Avec 2 paires → 4 équations pour 4 inconnues : **solution unique**.

In [ ]:
# ── Méthode algébrique directe ────────────────────────────────────────────
# Trouver 2 blocs tels que la matrice P soit inversible mod 26
from math import gcd

plain_blk1 = plain_blk2 = cipher_blk1 = cipher_blk2 = None
for i in range(0, len(TEST_PLAIN) - 2, 2):
    for j in range(i + 2, min(len(TEST_PLAIN), i + 40), 2):
        p1, p2 = TEST_PLAIN[i:i+2], TEST_PLAIN[j:j+2]
        P = [[ord(p1[0])-65, ord(p2[0])-65], [ord(p1[1])-65, ord(p2[1])-65]]
        det_P = (P[0][0]*P[1][1] - P[0][1]*P[1][0]) % 26
        if det_P != 0 and gcd(det_P, 26) == 1:
            plain_blk1, plain_blk2 = p1, p2
            cipher_blk1 = CIPHERTEXT[i:i+2]
            cipher_blk2 = CIPHERTEXT[j:j+2]
            break
    if plain_blk1:
        break

print(f"Paires connues (matrice P inversible mod 26) :")
print(f"  Clair  : {plain_blk1} + {plain_blk2}")
print(f"  Chiffré: {cipher_blk1} + {cipher_blk2}")

t0 = time.time()
recovered_K_direct = known_plaintext_attack(
    [plain_blk1, plain_blk2],
    [cipher_blk1, cipher_blk2]
)
t_direct = time.time() - t0

print(f"\nClé récupérée (algèbre) : {recovered_K_direct}")
print(f"Vraie clé               : {TRUE_KEY}")
print(f"Précision               : {key_accuracy(TRUE_KEY, recovered_K_direct):.0%}")
print(f"Temps                   : {t_direct*1000:.2f} ms")

In [ ]:
# ── Attaque CP-SAT à texte clair connu ───────────────────────────────────
plain_blocks  = [(ord(plain_blk1[0])-65,  ord(plain_blk1[1])-65),
                 (ord(plain_blk2[0])-65,  ord(plain_blk2[1])-65)]
cipher_blocks = [(ord(cipher_blk1[0])-65, ord(cipher_blk1[1])-65),
                 (ord(cipher_blk2[0])-65, ord(cipher_blk2[1])-65)]

print("Résolution CP-SAT (texte clair connu, 2 paires)...")
t0 = time.time()
cp_kp_result = solve_hill_known_plaintext(
    plain_blocks, cipher_blocks, time_limit=15.0
)
t_cp_kp = time.time() - t0

print(f"Statut    : {cp_kp_result['status']}  |  Temps: {t_cp_kp:.2f}s")
print(f"Clé trouvée : {cp_kp_result['key']}")
print(f"Vraie clé   : {TRUE_KEY}")
if cp_kp_result['key']:
    print(f"Précision   : {key_accuracy(TRUE_KEY, cp_kp_result['key']):.0%}")
    # Vérification : le déchiffrement doit fonctionner (padding possible si longueur impaire)
    plain_check = decrypt(CIPHERTEXT, cp_kp_result['key'])
    print(f"Déchiffrement OK : {plain_check[:len(TEST_PLAIN)] == TEST_PLAIN}")

In [ ]:
# ── Robustesse : combien de paires faut-il ? ──────────────────────────────
from math import gcd

print("Test avec 1 seule paire (sous-déterminé) :")
cp_1pair = solve_hill_known_plaintext(
    plain_blocks[:1], cipher_blocks[:1], time_limit=5.0
)
if cp_1pair['key']:
    print(f"  Clé trouvée : {cp_1pair['key']}  (exacte={cp_1pair['key']==TRUE_KEY})")
else:
    print(f"  Statut : {cp_1pair['status']}")

print("\nTest avec 2 paires indépendantes :")
# Prendre des blocs de positions différentes
pairs = []
for start in range(0, min(len(TEST_PLAIN), 20), 2):
    p = TEST_PLAIN[start:start+2]
    c = CIPHERTEXT[start:start+2]
    if len(p) == 2:
        pairs.append((p, c))

# Find 2 pairs where P matrix is invertible
found = False
for i, (p1, c1) in enumerate(pairs):
    for j, (p2, c2) in enumerate(pairs[i+1:]):
        P = [[ord(p1[0])-65, ord(p2[0])-65], [ord(p1[1])-65, ord(p2[1])-65]]
        det_P = (P[0][0]*P[1][1] - P[0][1]*P[1][0]) % 26
        if det_P != 0 and gcd(det_P, 26) == 1:
            print(f"  Paires : ('{p1}','{c1}') et ('{p2}','{c2}')")
            pb = [(ord(p1[0])-65, ord(p1[1])-65), (ord(p2[0])-65, ord(p2[1])-65)]
            cb = [(ord(c1[0])-65, ord(c1[1])-65), (ord(c2[0])-65, ord(c2[1])-65)]
            res = solve_hill_known_plaintext(pb, cb, time_limit=10.0)
            if res['key']:
                print(f"  Clé : {res['key']}  exacte={res['key']==TRUE_KEY}")
            found = True
            break
    if found:
        break

<a id='4'></a>
---
## 4. Attaque texte chiffré seul — CP-SAT bigrammes

### Modèle CP-SAT (texte chiffré seul)

On optimise directement la **matrice de déchiffrement** K_inv :

```
Variables   : kd[i][j] ∈ [0..25]  — matrice de déchiffrement

Pour chaque bloc b = (c0, c1) :
  p0[b] = (kd[0][0]*c0 + kd[0][1]*c1) mod 26   ← LINÉAIRE en kd (c0,c1 = constantes)
  p1[b] = (kd[1][0]*c0 + kd[1][1]*c1) mod 26

Contrainte d'invertibilité : gcd(det(K_inv), 26) = 1

Objectif : minimiser Σ bigram_cost(p0[b], p1[b])  +  Σ bigram_cost(p1[b], p0[b+1])
```

**Clé de l'approche** : puisque c0, c1 sont des constantes (lettres du texte chiffré),  
l'expression `kd[i][j]*c_j` est **linéaire** en `kd[i][j]` → le modèle complet reste linéaire !

In [ ]:
# ── CP-SAT texte chiffré seul ─────────────────────────────────────────────
print(f"Résolution CP-SAT (texte chiffré seul, {len(TEST_PLAIN)} lettres)...")
t0 = time.time()
cp_co_result = solve_hill_ciphertext_only(
    CIPHERTEXT, BIGRAM_LP,
    add_invertibility=True,
    time_limit=30.0
)
t_cp_co = time.time() - t0

print(f"Statut       : {cp_co_result['status']}  |  Temps: {t_cp_co:.2f}s")
print(f"K_inv trouvée: {cp_co_result['key_inv']}")
print(f"Vraie K_inv  : {_matrix_inv_mod26(TRUE_KEY)}")

if cp_co_result['key_inv']:
    found_kinv = cp_co_result['key_inv']
    true_kinv  = _matrix_inv_mod26(TRUE_KEY)
    acc = key_accuracy(true_kinv, found_kinv)
    print(f"Précision K_inv : {acc:.0%}")
    
    # Score bigrammes
    plain_score  = score_text(TEST_PLAIN,                  BIGRAM_LP)
    found_score  = score_text(cp_co_result['plaintext'],   BIGRAM_LP)
    print(f"Score bigrammes vrai texte  : {plain_score:.0f}")
    print(f"Score bigrammes trouvé      : {found_score:.0f}")
    print(f"\nDéchiffré (extrait) : {cp_co_result['plaintext'][:100]}")
    print(f"Vrai texte (extrait): {TEST_PLAIN[:100]}")

In [ ]:
# ── Impact de la taille du texte ──────────────────────────────────────────
TEXT_SIZES = [20, 50, 100, 200]
N_TRIALS   = 3
CP_LIMIT   = 20.0

base_text = clean_text(CORPUS) * 3
results_co = []

print("CP-SAT texte chiffré seul — impact de la taille du texte :")
for n in TEXT_SIZES:
    n_adj = n - n % 2  # even length for 2x2 blocks
    accs, times = [], []
    for trial in range(N_TRIALS):
        key  = generate_random_key()
        plain = base_text[:n_adj]
        cipher = encrypt(plain, key)
        t0 = time.time()
        res = solve_hill_ciphertext_only(cipher, BIGRAM_LP, time_limit=CP_LIMIT)
        elapsed = time.time() - t0
        if res['key_inv']:
            true_kinv = _matrix_inv_mod26(key)
            acc = key_accuracy(true_kinv, res['key_inv'])
        else:
            acc = 0.0
        accs.append(acc)
        times.append(elapsed)
    print(f"  n={n_adj:4d}  acc={np.mean(accs):.0%}  t={np.mean(times):.1f}s")
    results_co.append({'n': n_adj, 'acc': np.mean(accs), 'time': np.mean(times)})

<a id='5'></a>
---
## 5. Benchmark et comparaison

### Comparaison des deux approches

| Approche | Données requises | Garantie | Temps | Précision |
|----------|-----------------|----------|-------|-----------|
| Algèbre directe | 2 paires (clair, chiffré) | Exacte | < 1 ms | 100% |
| CP-SAT connu | 1–2 paires | Optimale | < 1 s | 100% |
| CP-SAT seul | Texte chiffré ≥ 100 lettres | Optimale (bigrammes) | 10–30 s | Variable |

In [ ]:
# ── Graphiques ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ns    = [r['n']    for r in results_co]
accs  = [r['acc']  for r in results_co]
times = [r['time'] for r in results_co]

ax1.plot(ns, [a*100 for a in accs],  'o-', color='steelblue', lw=2)
ax1.set_xlabel('Longueur du texte (lettres)')
ax1.set_ylabel('Précision K_inv (%)')
ax1.set_title('CP-SAT seul — précision vs taille du texte')
ax1.set_ylim(0, 105)
ax1.grid(True, alpha=0.3)

ax2.plot(ns, times, 's-', color='tomato', lw=2)
ax2.set_xlabel('Longueur du texte (lettres)')
ax2.set_ylabel('Temps (s)')
ax2.set_title('CP-SAT seul — temps vs taille du texte')
ax2.grid(True, alpha=0.3)

plt.suptitle('Benchmark — Hill cipher 2×2 (CP-SAT texte seul)', fontsize=13, y=1.02)
plt.tight_layout()
os.makedirs('../examples', exist_ok=True)
plt.savefig('../examples/benchmark_hill.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────────────────
print("=== Récapitulatif Hill cipher 2×2 ===")
print(f"{'Approche':<30} {'Données':<25} {'Temps':>10} {'Précision':>12}")
print('─' * 80)
print(f"{'Algèbre directe':<30} {'2 paires':<25} {t_direct*1000:>9.2f}ms {'100%':>12}")
print(f"{'CP-SAT connu (2 paires)':<30} {'2 paires':<25} {t_cp_kp:>9.2f}s  {'100%' if cp_kp_result['key']==TRUE_KEY else '?':>12}")
cp_sat_label = f"CP-SAT seul ({len(TEST_PLAIN)} lettres)"
cp_sat_prec = '100%' if (cp_co_result['key_inv'] and key_accuracy(_matrix_inv_mod26(TRUE_KEY), cp_co_result['key_inv']) >= 0.99) else '?'
print(f"{cp_sat_label:<30} {'texte chiffré':<25} {t_cp_co:>9.2f}s  {cp_sat_prec:>12}")

print("\n=== Conclusions ===")
print("""
Hill est un cas idéal pour le CP-SAT avec texte clair connu :
  - Les contraintes linéaires mod 26 sont directement encodables
  - 2 paires suffisent pour déterminer K de manière unique
  - CP-SAT prouve l'unicité de la solution

Sans texte clair connu, CP-SAT optimise le score bigrammes sur K_inv.
La clé : les produits kd[i][j]*c_j sont linéaires car c_j est constant.
→ Seul cipher avec un modèle PUREMENT LINÉAIRE dans CP-SAT !
""")

print("→ Suite : H3-5 — Évaluation comparée de tous les chiffrements")